# lecture 6 — metrics, KV cache

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

from util import CharacterTokenizer, Dataset
from gpt import GPT
from loss import estimate_loss
from metrics import Metrics

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
with open('dataset/processed/corpus_clean.txt', 'r') as f:
    text = f.read()

tokenizer = CharacterTokenizer(text)
print(f'vocab size: {len(tokenizer.vocab)}')

In [ ]:
batch_size   = 64
context_size = 256
n_embd       = 384
n_heads      = 6
n_layer      = 6
lr           = 3e-4

data_tensor = torch.tensor(tokenizer.encode(text), dtype=torch.long)
dataset = Dataset(data_tensor, context_size, batch_size)

print(f'train tokens: {len(dataset.train_data):,}')
print(f'val tokens  : {len(dataset.val_data):,}')

In [ ]:
model = GPT(
    len(tokenizer.vocab),
    n_embd=n_embd,
    context_size=context_size,
    n_head=n_heads,
    n_layer=n_layer,
).to(device)

model.load_state_dict(torch.load('luffy_gpt.pth', map_location=device))
print(f'params: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M')

# KV cache — faster inference

In [ ]:
import time

start_idx = torch.zeros((1, 1), dtype=torch.long, device=device)

t0 = time.time()
out = model.generate(start_idx, number_of_tokens=200, use_cache=False)
t1 = time.time()
print(f'without KV cache: {t1 - t0:.2f}s')

t0 = time.time()
out = model.generate(start_idx, number_of_tokens=200, use_cache=True)
t1 = time.time()
print(f'with KV cache   : {t1 - t0:.2f}s')

print('\n--- generated text ---')
print(tokenizer.decode(out[0].tolist()))

# metrics

In [ ]:
# perplexity = exp(cross_entropy_loss)
# lower is better — shows how surprised the model is by the next token
losses = estimate_loss(dataset, model)
print(f'train loss : {losses["train"]:.4f}  perplexity: {torch.exp(losses["train"]):.2f}')
print(f'val loss   : {losses["val"]:.4f}  perplexity: {torch.exp(losses["val"]):.2f}')

In [ ]:
# full metrics — ROUGE, BERTScore, masked accuracy
# this takes a few minutes (BERTScore loads a separate model)
metrics = Metrics(number_of_steps=5, mask_ratio=0.15)
results = metrics(dataset, model, tokenizer)

for k, v in results.items():
    print(f'{k:20s}: {v:.4f}')

# train with metrics reporting

In [ ]:
model2 = GPT(
    len(tokenizer.vocab),
    n_embd=n_embd,
    context_size=context_size,
    n_head=n_heads,
    n_layer=n_layer,
).to(device)

if torch.cuda.device_count() > 1:
    print(f'using {torch.cuda.device_count()} GPUs')
    model2 = nn.DataParallel(model2)

optimizer = torch.optim.AdamW(model2.parameters(), lr=lr)
metrics = Metrics(number_of_steps=5)

for step in range(5001):
    xb, yb = dataset.get_batch('train', device)
    _, loss = model2(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.mean().backward()
    optimizer.step()

    if step % 1000 == 0 or step == 5000:
        losses = estimate_loss(dataset, model2.module if isinstance(model2, nn.DataParallel) else model2)
        print(f'step {step:>5}  train: {losses["train"]:.4f}  val: {losses["val"]:.4f}')
        m = model2.module if isinstance(model2, nn.DataParallel) else model2
        results = metrics(dataset, m, tokenizer)
        for k, v in results.items():
            print(f'  {k:20s}: {v:.4f}')
        print()